# Part 2: Advanced Heuristics & Data Normalization

Beyond numeric tolerances, semantic validation frequently requires complex string mutation and schema alignment. This notebook demonstrates how to resolve systemic formatting discrepancies using Veridelta's declarative regular expressions and deterministic value mapping.

### 1. Simulating Systemic Formatting Discrepancies

During cross-platform migrations (e.g., legacy CRM to a modern SaaS platform), categorical enumerations and string formats routinely diverge. The following mock datasets simulate a scenario where the target system utilizes abbreviated tier codes and strips all non-numeric characters from phone numbers.

In [ ]:
import pathlib

import polars as pl

from veridelta import DataIngestor, DiffEngine, load_config

# Source System: Legacy CRM (Verbose statuses, formatted phone numbers)
src_df = pl.DataFrame(
    {
        "customer_id": ["CUST-801", "CUST-802"],
        "subscription_tier": ["Premium", "Enterprise"],
        "contact_number": ["(555) 123-4567", "+1 555-987-6543"],
    }
)

# Target System: Modern Platform (Abbreviated tiers, numeric-only phones)
tgt_df = pl.DataFrame(
    {
        "customer_id": ["CUST-801", "CUST-802"],
        "subscription_tier": ["PRM", "ENT"],
        "contact_number": ["5551234567", "15559876543"],
    }
)

# Serialize to disk to simulate remote data artifacts
src_df.write_parquet("legacy_crm.parquet")
tgt_df.write_parquet("modern_saas.parquet")
print("Mock migration artifacts serialized.")

# Output:
# Mock migration artifacts serialized.

### 2. Declarative Transformation Rules

Value mappings and regular expression replacements execute natively within the Polars Rust engine, ensuring optimal execution speed. These heuristics are declared within the configuration policy to maintain strict separation between validation rules and operational code.

In [ ]:
# Define translation heuristics via YAML configuration
yaml_config = """
primary_keys: ["customer_id"]
source:
  path: "legacy_crm.parquet"
  format: "parquet"
target:
  path: "modern_saas.parquet"
  format: "parquet"
rules:
  - column_names: ["subscription_tier"]
    value_map:                   # Deterministic translation dictionary
      "Premium": "PRM"
      "Enterprise": "ENT"
  - column_names: ["contact_number"]
    regex_replace:
      "[^0-9]": ""               # Strip all non-numeric characters prior to evaluation
"""
pathlib.Path("veridelta_heuristics.yaml").write_text(yaml_config)

# Execute validation pipeline
diff_cfg, src_cfg, tgt_cfg = load_config("veridelta_heuristics.yaml")

ingestor = DataIngestor(diff_cfg, src_cfg, tgt_cfg)
source_data, target_data = ingestor.get_dataframes()

engine = DiffEngine(diff_cfg, source_data, target_data)
summary = engine.run()

print(f"{' VERIDELTA EXECUTION SUMMARY ':=^50}")
print(f"Match Status:     {'SUCCESS' if summary.is_match else 'FAILED'}")
print(f"Total Rows:       {summary.total_rows_source:,}")
print(f"Semantic Diffs:   {summary.changed_count:,}")
print("=" * 50)

# Output:
# ========== VERIDELTA EXECUTION SUMMARY ===========
# Match Status:     SUCCESS
# Total Rows:       2
# Semantic Diffs:   0
# ==================================================

---
*(Run the cell below to clean up the temporary files generated by this notebook)*

In [ ]:
# Remove local artifacts (housekeeping)
for f in ["legacy_crm.parquet", "modern_saas.parquet", "veridelta_heuristics.yaml"]:
    pathlib.Path(f).unlink(missing_ok=True)